# ECG classification - V4

**V4 = V3 with balancing up-sampling -> SMOTE (one change).**
- *Prior result (V3):* FB is the weakest class (sensitivity 0.75). FB has only 642 training beats,
  copied ~31x to reach 20,000, so the model sees the same points again and again (overfits them).
- *Theory:* SMOTE creates new points between minority neighbours instead of copies, widening the
  minority decision region.
- *Paper:* Chawla et al., JAIR 2002 (SMOTE); proposed in Sec. VIII of our report.

In [ ]:
!pip -q install wfdb peakutils PyWavelets imbalanced-learn lightgbm

In [ ]:
import os; os.makedirs('src', exist_ok=True)
open('src/__init__.py','w').write('')

In [ ]:
%%writefile src/config.py
"""Experiment versions. Each version changes exactly ONE factor vs the version it is based on."""

VERSIONS = {
    # Reproduction of the original repo / paper (Subba & Chingtham, 2024)
    "V0": dict(based_on=None, features=["base"], model="rf", balance="upsample",
               change="baseline: paper features + RF + up-sampling to 20k"),
    "V1": dict(based_on="V0", features=["base", "rr"], model="rf", balance="upsample",
               change="+ RR-interval features (pre, post, local mean, ratio)"),
    "V2": dict(based_on="V1", features=["base", "rr", "morph"], model="rf", balance="upsample",
               change="+ time-domain morphology (beat waveform pooled to 72 points)"),
    "V3": dict(based_on="V2", features=["base", "rr", "morph"], model="lgbm", balance="upsample",
               change="classifier RF -> LightGBM (gradient boosting)"),
    "V4": dict(based_on="V3", features=["base", "rr", "morph"], model="lgbm", balance="smote",
               change="balancing: random up-sampling -> SMOTE"),
    "V5": dict(based_on="V4", features=["base", "rr", "morph", "rr_norm"], model="lgbm",
               balance="smote", change="+ patient-normalised RR (RR / record mean RR)"),
}


In [ ]:
%%writefile src/dataset.py
"""Load MIT-BIH beats, AAMI labels, RR intervals, and train/test splits."""
from __future__ import annotations

import glob
import logging
import os
from dataclasses import dataclass

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

log = logging.getLogger(__name__)

AAMI_CLASSES = ("N", "SVEB", "VEB", "FB", "Q")
SYMBOL_TO_AAMI = {
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    "A": "SVEB", "a": "SVEB", "J": "SVEB", "S": "SVEB",
    "V": "VEB", "E": "VEB",
    "F": "FB",
    "/": "Q", "f": "Q", "Q": "Q",
}
CLASS_TO_INT = {c: i for i, c in enumerate(AAMI_CLASSES)}
FS = 360

# de Chazal et al. (2004) inter-patient split. Paced records 102, 104, 107, 217 are excluded.
DS1 = [101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124, 201, 203, 205, 207,
       208, 209, 215, 220, 223, 230]
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212, 213, 214, 219, 221,
       222, 228, 231, 232, 233, 234]


@dataclass
class Beats:
    X: np.ndarray    # (n, window) beat segments
    y: np.ndarray    # (n,) int labels
    rr: np.ndarray   # (n, 4) RR features in seconds: pre, post, local mean, pre/local
    rec: np.ndarray  # (n,) record number


def find_data_dir(hint: str | None = None) -> str:
    """Return a folder holding the WFDB records (searches /kaggle/input if no hint)."""
    if hint and glob.glob(os.path.join(hint, "*.hea")):
        return hint
    roots = [hint] if hint else ["/kaggle/input", "./data", "."]
    for root in roots:
        hits = glob.glob(os.path.join(root, "**", "100.hea"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError("MIT-BIH records (100.hea ...) not found")


def _rr_features(r_pos: np.ndarray) -> np.ndarray:
    """RR features for each beat from the R-peak sample positions of one record."""
    rr = np.diff(r_pos) / FS
    pre = np.concatenate([[rr[0]], rr])
    post = np.concatenate([rr, [rr[-1]]])
    # local mean of the previous 10 pre-RR intervals (causal, includes the current one)
    csum = np.cumsum(np.concatenate([[0.0], pre]))
    idx = np.arange(len(pre))
    lo = np.maximum(0, idx - 9)
    local = (csum[idx + 1] - csum[lo]) / (idx + 1 - lo)
    return np.stack([pre, post, local, pre / local], axis=1)


def load_beats(data_dir: str, window: int = 360, preprocessor=None) -> Beats:
    import wfdb

    names = sorted(os.path.splitext(f)[0] for f in os.listdir(data_dir)
                   if f.endswith(".hea") and f[:3].isdigit())
    half = window // 2
    Xs, ys, rrs, recs = [], [], [], []
    for name in names:
        path = os.path.join(data_dir, name)
        rec = wfdb.rdrecord(path)
        ann = wfdb.rdann(path, "atr")
        sig = rec.p_signal[:, 0].astype(np.float64)
        if preprocessor is not None:
            sig = preprocessor(sig)
        keep = [(s, SYMBOL_TO_AAMI[c]) for s, c in zip(ann.sample, ann.symbol) if c in SYMBOL_TO_AAMI]
        pos = np.array([s for s, _ in keep])
        rr = _rr_features(pos)
        for (s, cls), r in zip(keep, rr):
            a, b = s - half, s - half + window
            if a < 0 or b > len(sig):
                continue
            Xs.append(sig[a:b]); ys.append(CLASS_TO_INT[cls]); rrs.append(r); recs.append(int(name))
    beats = Beats(np.asarray(Xs), np.asarray(ys), np.asarray(rrs), np.asarray(recs))
    log.info("Loaded %d beats from %d records", len(beats.y), len(names))
    return beats


def split_indices(beats: Beats, mode: str = "random", seed: int = 42, test_size: float = 0.2):
    """'random' = paper protocol (stratified 80/20 over beats). 'interpatient' = DS1/DS2."""
    idx = np.arange(len(beats.y))
    if mode == "random":
        return train_test_split(idx, test_size=test_size, random_state=seed, stratify=beats.y)
    if mode == "interpatient":
        return idx[np.isin(beats.rec, DS1)], idx[np.isin(beats.rec, DS2)]
    raise ValueError(mode)


def upsample(F: np.ndarray, y: np.ndarray, target: int = 20_000, seed: int = 42):
    """Paper balancing: each minority class up-sampled with replacement to `target`."""
    classes, counts = np.unique(y, return_counts=True)
    major = classes[np.argmax(counts)]
    Fp, yp = [], []
    for c in classes:
        Fc = F[y == c]
        if c != major and len(Fc) < target:
            Fc = resample(Fc, replace=True, n_samples=target, random_state=seed)
        Fp.append(Fc); yp.append(np.full(len(Fc), c))
    F2, y2 = np.vstack(Fp), np.concatenate(yp)
    perm = np.random.default_rng(seed).permutation(len(y2))
    return F2[perm], y2[perm]


def smote(F: np.ndarray, y: np.ndarray, target: int = 20_000, seed: int = 42):
    """SMOTE: synthesise minority samples (k=5 neighbours) up to `target` per minority class."""
    from imblearn.over_sampling import SMOTE

    classes, counts = np.unique(y, return_counts=True)
    major = classes[np.argmax(counts)]
    strategy = {int(c): max(int(n), target) for c, n in zip(classes, counts) if c != major}
    return SMOTE(sampling_strategy=strategy, k_neighbors=5, random_state=seed).fit_resample(F, y)


In [ ]:
%%writefile src/preprocess.py
"""
preprocess.py
=============
Pre-processing module for ECG signals.

Two stages, exactly as described in the paper (Section III-C):

1. Wavelet denoising
   A 9-level ``bior4.4`` wavelet decomposition. Coefficients from index 0 up to
   ``cutoff_low`` and from ``cutoff_high`` to the end are zeroed out, which
   removes the corresponding (lowest- and highest-frequency) sub-bands before
   the signal is reconstructed.

2. Baseline fitting / removal
   Multiple median filtration of different widths estimates the slowly varying
   baseline (wander), which is then subtracted, keeping only the signal of
   interest.
"""

from __future__ import annotations

import logging
from typing import Sequence

import numpy as np
import pywt
from scipy.signal import medfilt

logger = logging.getLogger(__name__)

DEFAULT_WAVELET = "bior4.4"
DEFAULT_LEVEL = 9


def wavelet_denoise(
    signal: np.ndarray,
    wavelet: str = DEFAULT_WAVELET,
    level: int = DEFAULT_LEVEL,
    cutoff_low: int = 1,
    cutoff_high: int = 1,
) -> np.ndarray:
    """Denoise a 1-D ECG signal via band-limited wavelet reconstruction.

    The signal is decomposed with ``pywt.wavedec`` into a coefficient list
    ``[cA_n, cD_n, cD_{n-1}, ..., cD_1]`` (low frequency first, high frequency
    last). The first ``cutoff_low`` coefficient arrays (lowest frequencies,
    i.e. baseline / very-low-frequency content) and the last ``cutoff_high``
    coefficient arrays (highest frequencies, i.e. high-frequency noise) are set
    to zero. The remaining mid-band coefficients are kept and the signal is
    reconstructed with ``pywt.waverec``.

    Parameters
    ----------
    signal : np.ndarray
        1-D input signal.
    wavelet : str, default 'bior4.4'
        Mother wavelet.
    level : int, default 9
        Number of decomposition levels.
    cutoff_low : int, default 1
        Number of leading (lowest-frequency) coefficient arrays to zero out.
    cutoff_high : int, default 1
        Number of trailing (highest-frequency) coefficient arrays to zero out.

    Returns
    -------
    np.ndarray
        The denoised, length-matched signal.
    """
    signal = np.asarray(signal, dtype=np.float64)
    original_length = signal.shape[0]

    # Clamp the requested level to what the signal length actually supports.
    max_level = pywt.dwt_max_level(original_length, pywt.Wavelet(wavelet).dec_len)
    used_level = min(level, max_level)

    coeffs = pywt.wavedec(signal, wavelet, level=used_level)
    n = len(coeffs)

    # Zero the lowest-frequency arrays (0 .. cutoff_low) ...
    low = max(0, min(cutoff_low, n))
    for i in range(low):
        coeffs[i] = np.zeros_like(coeffs[i])

    # ... and the highest-frequency arrays (cutoff_high .. end).
    high = max(0, min(cutoff_high, n))
    for i in range(n - high, n):
        coeffs[i] = np.zeros_like(coeffs[i])

    reconstructed = pywt.waverec(coeffs, wavelet)
    # waverec may return a signal 1 sample longer due to padding; trim it.
    return reconstructed[:original_length]


def median_baseline_removal(
    signal: np.ndarray,
    widths: Sequence[int] = (71, 215),
) -> np.ndarray:
    """Estimate and subtract the baseline using successive median filters.

    A cascade of median filters with increasing (odd) widths progressively
    removes the QRS complexes, P and T waves, leaving an estimate of the
    low-frequency baseline wander. Subtracting that estimate from the original
    signal corrects the baseline.

    Parameters
    ----------
    signal : np.ndarray
        1-D input signal.
    widths : sequence of int, default (71, 215)
        Odd median-filter window widths applied in sequence. At 360 Hz these
        correspond to roughly 200 ms and 600 ms windows. Even values are
        incremented by 1 to stay odd (required by ``scipy.signal.medfilt``).

    Returns
    -------
    np.ndarray
        The baseline-corrected signal.
    """
    signal = np.asarray(signal, dtype=np.float64)
    baseline = signal.copy()
    for w in widths:
        kernel = w if w % 2 == 1 else w + 1
        kernel = min(kernel, _largest_odd_le(len(baseline)))
        if kernel < 1:
            continue
        baseline = medfilt(baseline, kernel_size=kernel)
    return signal - baseline


def _largest_odd_le(n: int) -> int:
    """Largest odd integer <= n (median kernels must be odd and <= signal len)."""
    if n <= 0:
        return 1
    return n if n % 2 == 1 else n - 1


def preprocess_signal(
    signal: np.ndarray,
    wavelet: str = DEFAULT_WAVELET,
    level: int = DEFAULT_LEVEL,
    cutoff_low: int = 1,
    cutoff_high: int = 1,
    median_widths: Sequence[int] = (71, 215),
) -> np.ndarray:
    """Full single-signal pre-processing: wavelet denoise -> baseline removal."""
    denoised = wavelet_denoise(
        signal, wavelet=wavelet, level=level,
        cutoff_low=cutoff_low, cutoff_high=cutoff_high,
    )
    return median_baseline_removal(denoised, widths=median_widths)


def preprocess_batch(
    signals: np.ndarray,
    **kwargs,
) -> np.ndarray:
    """Apply :func:`preprocess_signal` to every row of a 2-D array of beats."""
    signals = np.asarray(signals, dtype=np.float64)
    out = np.empty_like(signals)
    for i in range(signals.shape[0]):
        out[i] = preprocess_signal(signals[i], **kwargs)
    logger.info("Pre-processed %d signals.", signals.shape[0])
    return out


In [ ]:
%%writefile src/feature_extraction.py
"""
feature_extraction.py
======================
Feature extraction module (paper Section III-F).

For every (pre-processed) heartbeat segment the following features are computed
and concatenated into a single fixed-length feature vector:

Amplitude / temporal / statistical features
--------------------------------------------
* R-peaks detected with ``peakutils.indexes(thres=0.5, min_dist=100)``.
* QRS duration estimated from the differences (``numpy.diff``) of R-peak
  locations.
* Statistical metrics of the R-peak amplitudes: mean, median, sum, std.

High-Order Statistics (HOS)
---------------------------
* Skewness  = sum((x_i - mean)^3) / (n * sigma^3)
* Kurtosis  = sum((x_i - mean)^4) / (n * sigma^4) - 3   (excess kurtosis)

Frequency-domain features
--------------------------
* Magnitude spectrum from the Fast Fourier Transform (FFT) of the signal.

The scalar features are placed first, followed by the FFT magnitude spectrum,
giving every beat a feature vector of identical length.
"""

from __future__ import annotations

import logging
from typing import List

import numpy as np
import peakutils  # noqa: E402

logger = logging.getLogger(__name__)

# Number of scalar (non-spectral) features produced per beat. Kept as a module
# constant so downstream code can reason about the feature layout if needed.
N_SCALAR_FEATURES = 9


def skewness(x: np.ndarray) -> float:
    """Skewness = sum((x_i - mean)^3) / (n * sigma^3)  (paper eq. 2)."""
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    if n == 0:
        return 0.0
    mean = x.mean()
    sigma = x.std()  # population standard deviation
    if sigma == 0:
        return 0.0
    return float(np.sum((x - mean) ** 3) / (n * sigma ** 3))


def kurtosis(x: np.ndarray) -> float:
    """Excess kurtosis = sum((x_i - mean)^4) / (n * sigma^4) - 3  (paper eq. 3)."""
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    if n == 0:
        return 0.0
    mean = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    return float(np.sum((x - mean) ** 4) / (n * sigma ** 4) - 3.0)


def detect_r_peaks(signal: np.ndarray, thres: float = 0.5, min_dist: int = 100) -> np.ndarray:
    """Return the indices of R-peaks using peakutils with the paper's settings."""
    signal = np.asarray(signal, dtype=np.float64)
    # peakutils expects a normalised threshold in [0, 1] relative to the signal
    # amplitude range, which is exactly how the paper uses threshold=0.5.
    return peakutils.indexes(signal, thres=thres, min_dist=min_dist)


def _rpeak_amplitude_stats(signal: np.ndarray, peak_idx: np.ndarray) -> List[float]:
    """mean, median, sum, std of the R-peak amplitudes (0 if no peaks)."""
    if peak_idx.size == 0:
        return [0.0, 0.0, 0.0, 0.0]
    amps = signal[peak_idx]
    return [
        float(np.mean(amps)),
        float(np.median(amps)),
        float(np.sum(amps)),
        float(np.std(amps)),
    ]


def _qrs_duration_stats(peak_idx: np.ndarray) -> List[float]:
    """mean and std of the inter-R-peak intervals (proxy for QRS timing).

    QRS duration is estimated from numpy.diff() of the R-peak positions, as the
    paper specifies. With fewer than two peaks the interval is undefined -> 0.
    """
    if peak_idx.size < 2:
        return [0.0, 0.0]
    intervals = np.diff(peak_idx).astype(np.float64)
    return [float(np.mean(intervals)), float(np.std(intervals))]


def fft_magnitude(signal: np.ndarray) -> np.ndarray:
    """Magnitude spectrum (one-sided) of the FFT of ``signal`` (paper eq. 5)."""
    signal = np.asarray(signal, dtype=np.float64)
    spectrum = np.fft.rfft(signal)
    return np.abs(spectrum)


def extract_features(signal: np.ndarray, thres: float = 0.5, min_dist: int = 100) -> np.ndarray:
    """Compute the full feature vector for a single beat segment.

    Layout
    ------
    [ n_rpeaks,
      amp_mean, amp_median, amp_sum, amp_std,
      qrs_mean, qrs_std,
      skewness, kurtosis,
      <FFT magnitude spectrum ...> ]
    """
    signal = np.asarray(signal, dtype=np.float64)

    peak_idx = detect_r_peaks(signal, thres=thres, min_dist=min_dist)

    scalar_features: List[float] = []
    scalar_features.append(float(peak_idx.size))               # number of R-peaks
    scalar_features.extend(_rpeak_amplitude_stats(signal, peak_idx))  # 4 stats
    scalar_features.extend(_qrs_duration_stats(peak_idx))      # 2 stats
    scalar_features.append(skewness(signal))                   # HOS
    scalar_features.append(kurtosis(signal))                   # HOS

    spectral_features = fft_magnitude(signal)

    return np.concatenate([np.asarray(scalar_features, dtype=np.float64), spectral_features])


def extract_features_batch(
    signals: np.ndarray,
    thres: float = 0.5,
    min_dist: int = 100,
) -> np.ndarray:
    """Run :func:`extract_features` over every row of a 2-D array of beats."""
    signals = np.asarray(signals, dtype=np.float64)
    features = [extract_features(sig, thres=thres, min_dist=min_dist) for sig in signals]
    matrix = np.vstack(features)
    logger.info("Extracted feature matrix of shape %s.", matrix.shape)
    return matrix


In [ ]:
%%writefile src/features.py
"""Feature groups. 'base' = original paper features; others are added one per version."""
from __future__ import annotations

import numpy as np

from . import feature_extraction as fe


def base(X: np.ndarray, **_) -> np.ndarray:
    """Paper features: R-peak stats, QRS proxy, skewness, kurtosis, FFT magnitude (190 dims)."""
    return fe.extract_features_batch(X)


def rr(X: np.ndarray, rr: np.ndarray, **_) -> np.ndarray:
    """RR-interval features: pre-RR, post-RR, local mean RR, pre/local ratio (4 dims)."""
    return rr


def morph(X: np.ndarray, pool: int = 5, **_) -> np.ndarray:
    """Time-domain morphology: the beat waveform mean-pooled by `pool` (360/5 = 72 dims)."""
    n, w = X.shape
    return X[:, : w - w % pool].reshape(n, -1, pool).mean(axis=2)


def rr_norm(X: np.ndarray, rr: np.ndarray, rec: np.ndarray, **_) -> np.ndarray:
    """Patient-normalised RR: pre, post, local RR divided by the record's mean RR (3 dims)."""
    out = np.empty((len(rr), 3))
    for r in np.unique(rec):
        m = rec == r
        out[m] = rr[m, :3] / rr[m, 0].mean()
    return out


GROUPS = {"base": base, "rr": rr, "morph": morph, "rr_norm": rr_norm}


In [ ]:
%%writefile src/models.py
"""Classifiers."""
from __future__ import annotations

from sklearn.ensemble import RandomForestClassifier


def build(name: str, seed: int = 42):
    if name == "rf":  # paper hyper-parameters
        return RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2,
                                      min_samples_leaf=1, max_features="sqrt",
                                      random_state=seed, n_jobs=-1)
    if name == "lgbm":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              random_state=seed, n_jobs=-1, verbose=-1)
    raise ValueError(name)


In [ ]:
%%writefile src/metrics.py
"""Per-class and overall metrics (one-vs-rest from the confusion matrix)."""
from __future__ import annotations

import numpy as np
from sklearn.metrics import confusion_matrix


def evaluate(y_true, y_pred, class_names):
    labels = list(range(len(class_names)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    total = cm.sum()
    per = {}
    for i, c in enumerate(class_names):
        tp = cm[i, i]; fn = cm[i].sum() - tp; fp = cm[:, i].sum() - tp; tn = total - tp - fn - fp
        se = tp / (tp + fn) if tp + fn else 0.0
        sp = tn / (tn + fp) if tn + fp else 0.0
        ppv = tp / (tp + fp) if tp + fp else 0.0
        f1 = 2 * se * ppv / (se + ppv) if se + ppv else 0.0
        per[c] = {"support": int(cm[i].sum()), "sensitivity": float(se), "specificity": float(sp),
                  "ppv": float(ppv), "f1": float(f1)}
    present = [c for c in class_names if per[c]["support"] > 0]
    return {
        "accuracy": float(np.trace(cm) / total),
        "macro_f1": float(np.mean([per[c]["f1"] for c in present])),
        "macro_sensitivity": float(np.mean([per[c]["sensitivity"] for c in present])),
        "macro_ppv": float(np.mean([per[c]["ppv"] for c in present])),
        "per_class": per,
        "confusion": cm.tolist(),
    }


def to_text(m, class_names) -> str:
    lines = [f"accuracy {m['accuracy']:.4f} | macro-F1 {m['macro_f1']:.4f} | "
             f"macro-Se {m['macro_sensitivity']:.4f} | macro-PPV {m['macro_ppv']:.4f}", "",
             f"{'class':<6}{'n':>7}{'Se':>8}{'Sp':>8}{'PPV':>8}{'F1':>8}"]
    for c in class_names:
        p = m["per_class"][c]
        lines.append(f"{c:<6}{p['support']:>7}{p['sensitivity']:>8.4f}{p['specificity']:>8.4f}"
                     f"{p['ppv']:>8.4f}{p['f1']:>8.4f}")
    lines += ["", "confusion (rows=true, cols=pred): " + " ".join(class_names)]
    lines += [f"{c:<6}" + "".join(f"{v:>7}" for v in row) for c, row in zip(class_names, m["confusion"])]
    return "\n".join(lines)


In [ ]:
%%writefile src/run.py
"""Run one or more versions:  python -m src.run --versions V0 V1 --split random --out results"""
from __future__ import annotations

import argparse
import json
import logging
import os
import time

import numpy as np

from . import dataset as ds, features, metrics, models, preprocess as pp
from .config import VERSIONS

log = logging.getLogger("run")


def prep(sig):
    return pp.preprocess_signal(sig, level=9, cutoff_low=1, cutoff_high=1, median_widths=(71, 215))


def run_version(name, beats, feats_cache, tr, te, split, out_root, seed=42):
    cfg = VERSIONS[name]
    for g in cfg["features"]:
        if g not in feats_cache:
            feats_cache[g] = features.GROUPS[g](beats.X, rr=beats.rr, rec=beats.rec)
    F = np.hstack([feats_cache[g] for g in cfg["features"]])
    Ftr, ytr = F[tr], beats.y[tr]
    if cfg["balance"] == "upsample":
        Ftr, ytr = ds.upsample(Ftr, ytr, seed=seed)
    elif cfg["balance"] == "smote":
        Ftr, ytr = ds.smote(Ftr, ytr, seed=seed)
    t0 = time.time()
    model = models.build(cfg["model"], seed=seed)
    model.fit(Ftr, ytr)
    pred = model.predict(F[te])
    m = metrics.evaluate(beats.y[te], pred, list(ds.AAMI_CLASSES))
    m.update(version=name, split=split, n_features=int(F.shape[1]), n_train=int(len(ytr)),
             n_test=int(len(te)), train_seconds=round(time.time() - t0, 1), config=cfg)
    out = os.path.join(out_root, name, split)
    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, "metrics.json"), "w") as f:
        json.dump(m, f, indent=2)
    txt = f"{name} [{split}] {cfg['change']}\n" + metrics.to_text(m, list(ds.AAMI_CLASSES))
    with open(os.path.join(out, "report.txt"), "w") as f:
        f.write(txt + "\n")
    print(txt + "\n", flush=True)
    return m


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--versions", nargs="+", default=list(VERSIONS))
    ap.add_argument("--split", nargs="+", default=["random"], choices=["random", "interpatient"])
    ap.add_argument("--data-dir", default=None)
    ap.add_argument("--out", default="results")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s: %(message)s")

    beats = ds.load_beats(ds.find_data_dir(a.data_dir), preprocessor=prep)
    cache = {}
    rows = []
    for split in a.split:
        tr, te = ds.split_indices(beats, split, seed=a.seed)
        for v in a.versions:
            m = run_version(v, beats, cache, tr, te, split, a.out, seed=a.seed)
            pc = m["per_class"]
            rows.append([v, split, m["accuracy"], m["macro_f1"], pc["SVEB"]["sensitivity"],
                         pc["SVEB"]["f1"], pc["VEB"]["f1"], pc["FB"]["sensitivity"], pc["FB"]["f1"]])
    os.makedirs(a.out, exist_ok=True)
    path = os.path.join(a.out, "summary_" + "_".join(a.split) + ".csv")
    with open(path, "w") as f:
        f.write("version,split,accuracy,macro_f1,sveb_se,sveb_f1,veb_f1,fb_se,fb_f1\n")
        for r in rows:
            f.write(",".join([r[0], r[1]] + [f"{x:.4f}" for x in r[2:]]) + "\n")
    print(open(path).read())


if __name__ == "__main__":
    main()


In [ ]:
!python -m src.run --versions V4 --split random interpatient --out results

In [ ]:
import glob
for p in sorted(glob.glob('results/**/report.txt', recursive=True)):
    print(open(p).read())